## 根据文档，编写从相机摄取不同角度图片的代码

1、设置迭代的终止条件
2、定义图片中角点存贮的数组

In [4]:
import numpy as np
import cv2 as cv
import glob
 
# termination criteria
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((7*7,3), np.float32)
objp[:,:2] = np.mgrid[0:7,0:7].T.reshape(-1,2)
objp=0.008*objp

导入图片，分析角点。并画出

In [6]:
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.
 
images = glob.glob('*.jpg')
 
for fname in images:
    img = cv.imread(fname)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Find the chess board corners
    ret, corners = cv.findChessboardCorners(gray, (7,7), None)
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
        corners2 = cv.cornerSubPix(gray,corners, (5,5), (-1,-1), criteria)
        imgpoints.append(corners2)
    # Draw and display the corners
    cv.drawChessboardCorners(img, (7,7), corners2, ret)
    cv.imshow('img', img)
    cv.waitKey(500)

cv.destroyAllWindows()

In [7]:
images

['calibresult.jpg']

获取是否有返回值、内参、畸变、旋转向量和平移向量，具体原理可查询相关论文。

In [8]:
ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

内参

In [9]:
mtx

array([[553.16663609,   0.        , 214.68788029],
       [  0.        , 934.97498965, 155.16856768],
       [  0.        ,   0.        ,   1.        ]])

畸变参数

In [10]:
dist

array([[-2.92961177e-01, -6.56635595e-02, -6.39604380e-03,
         2.53313462e-02,  5.37462016e+01]])

用获取的内参和畸变参数，对第一张图片进行畸变矫正。

In [11]:
img = cv.imread(images[0])
h, w = img.shape[:2]
newcameramtx, roi = cv.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

In [12]:
# undistort
dst = cv.undistort(img, mtx, dist, None, newcameramtx)
# crop the image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv.imwrite('calibresult.png', dst)

True

In [13]:
images[0]

'calibresult.jpg'

了解rvecs, tvecs的作用

In [14]:
def draw(img, corners, imgpts):
 corner = tuple(corners[0].ravel().astype(int))
 img = cv.line(img, corner , tuple(imgpts[0].ravel().astype(int)), (255,0,0), 5)
 img = cv.line(img, corner, tuple(imgpts[1].ravel().astype(int)), (0,255,0), 5)
 img = cv.line(img, corner , tuple(imgpts[2].ravel().astype(int)), (0,0,255), 5)
 return img

In [15]:
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)
objp = np.zeros((7*7,3), np.float32)
objp[:,:2] = np.mgrid[0:7,0:7].T.reshape(-1,2)
 
axis = np.float32([[3,0,0], [0,3,0], [0,0,-3]]).reshape(-1,3)

In [ ]:
 img = cv.imread(images[0])
 gray = cv.cvtColor(img,cv.COLOR_BGR2GRAY)
 ret, corners = cv.findChessboardCorners(gray, (7,7),None)
 
 if ret == True:
     corners2 = cv.cornerSubPix(gray,corners,(11,11),(-1,-1),criteria)
 
 # Find the rotation and translation vectors.
 ret,rvecs, tvecs = cv.solvePnP(objp, corners2, mtx, dist)
 
 # project 3D points to image plane
 imgpts, jac = cv.projectPoints(axis, rvecs, tvecs, mtx, dist)
 
 img = draw(img,corners2 ,imgpts)
 cv.imshow('img',img)
 k = cv.waitKey(0) & 0xFF
 if k == ord('s'):
     cv.imwrite(fname[:6]+'.png', img)
 
cv.destroyAllWindows()

In [15]:
rvecs, tvecs

(array([[-0.24406926],
        [ 0.00712551],
        [ 0.16918039]]),
 array([[-3.82911151],
        [-3.74079138],
        [22.01701187]]))